In [3]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 5.1 MB/s  0:00:00m 5.1 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install pandas
# Just run the code above!


SyntaxError: invalid syntax (3901193241.py, line 1)

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            confusion_matrix, classification_report, roc_auc_score)
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 1. FLEXIBLE FILE LOADER (Works Everywhere)
# ========================================
def load_dataset():
    """Smart CSV finder - handles Jupyter, VSCode, Terminal"""
    
    # For Jupyter - check attached files first
    try:
        df = pd.read_csv('ecg_cvd_dataset_expanded.csv')
        print("✅ Found: ecg_cvd_dataset_expanded.csv (same folder)")
        return df
    except:
        pass
    
    # Check Downloads folder
    try:
        df = pd.read_csv('Downloads/ecg_cvd_dataset_expanded.csv')
        print("✅ Found: Downloads/ecg_cvd_dataset_expanded.csv")
        return df
    except:
        pass
    
    # List all CSV files to help you find it
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    print("🔍 CSV files found:", csv_files)
    
    # Try first CSV file found
    if csv_files:
        df = pd.read_csv(csv_files[0])
        print(f"✅ Auto-loaded: {csv_files[0]}")
        return df
    
    raise FileNotFoundError(
        "\n❌ CSV NOT FOUND! Quick fixes:\n"
        "1. Drag CSV into this folder/Jupyter\n"
        "2. Run: pd.read_csv('YOUR_EXACT_PATH.csv')\n"
        "3. Or rename file to: ecg_cvd_dataset_expanded.csv"
    )

# Load data
print("🔥 Loading ECG_CVD Dataset...")
df = load_dataset()

print(f"✅ Dataset loaded: {df.shape}")
print("\nTarget classes:", df['target_class'].value_counts().sort_index())

# ========================================
# 2. DATA PREPROCESSING
# ========================================
print("\n🧹 Preprocessing...")

# Clean gender column
df['gender'] = df['gender'].map({'M': 1, 'F': 0}).fillna(0)

# 19 Key Features
feature_cols = [
    'heart_rate_bpm', 'rr_mean_ms', 'rr_std_ms', 'sdnn_ms', 'rmssd_ms',
    'p_wave_duration_ms', 'qrs_duration_ms', 'qt_interval_ms', 'st_elevation_mv',
    't_wave_amplitude_mv', 'lf_power', 'hf_power', 'lf_hf_ratio', 'spectral_entropy',
    'age', 'gender', 'bp_systolic', 'bp_diastolic', 'cholesterol'
]

X = df[feature_cols].fillna(0)  # Fill any missing values
y_str = df['target_class'].astype(str)

# Encode labels: 0=Normal, 1=Arrhythmia, 2=CHD, 3=Cardiomyopathy, 4=Stroke, 5=Heart Failure
le = LabelEncoder()
y = le.fit_transform(y_str)

class_names = ['Normal', 'Arrhythmia', 'CHD', 'Cardiomyopathy', 'Stroke', 'Heart Failure']

# 60/20/20 Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"✅ Train: {X_train.shape}, Test: {X_test.shape}")

# ========================================
# 3. YOUR 4 MODELS (NO XGBoost - Mac Fixed)
# ========================================
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# ========================================
# 4. EVALUATION ENGINE
# ========================================
def evaluate_model(model, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    }
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"✅ {name:>18}: F1={metrics['F1-Score']:.3f}")
    
    return metrics, cm, y_pred

# ========================================
# 5. TRAIN & COMPARE
# ========================================
print("\n" + "="*60)
print("🚀 TRAINING YOUR 4 MODELS")
print("="*60)

results = {}
cms = {}

for name, model in models.items():
    metrics, cm, preds = evaluate_model(model, name)
    results[name] = metrics
    cms[name] = cm

# Results Table
results_df = pd.DataFrame(results).T.round(4)
print("\n" + "="*80)
print("📊 FINAL RESULTS - YOUR THESIS TABLE")
print("="*80)
print(results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']])

# ========================================
# 6. PUBLICATION PLOTS
# ========================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

metrics = ['Accuracy', 'Precision', 'F1-Score', 'ROC-AUC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for i, metric in enumerate(metrics):
    scores = [results[model][metric] for model in models]
    axes[i].bar(models.keys(), scores, color=colors, alpha=0.8, edgecolor='black')
    axes[i].set_title(metric, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('ECG-CVD Baseline Models (Your Thesis Chapter 3)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('thesis_baseline_results.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================================
# 7. BEST MODEL DEEP DIVE
# ========================================
best_model = results_df['F1-Score'].idxmax()
best_f1 = results_df.loc[best_model, 'F1-Score']

print(f"\n🏆 BEST: {best_model} (F1: {best_f1:.3f})")

plt.figure(figsize=(10, 8))
sns.heatmap(sms[best_model], annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'{best_model} Confusion Matrix\nF1-Score: {best_f1:.3f}', fontweight='bold')
plt.ylabel('True'), plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('best_model_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# ========================================
# 8. THESIS SUMMARY
# ========================================
print("\n" + "="*80)
print("🎓 THESIS CHAPTER 3 - COMPLETE!")
print("="*80)
print(f"✅ Dataset: {len(df):,} ECG records")
print(f"✅ Features: 19 ECG + clinical")
print(f"✅ Classes: 6 CVD conditions")
print(f"✅ Best F1: {best_f1:.1%} ({best_model})")
print(f"✅ Perfect baseline → Justifies DL chapter!")
print("\n📄 SAVED FOR THESIS:")
print("- thesis_baseline_results.png")
print("- best_model_matrix.png") 
print("- results_df (copy to Excel)")

print("\n🎯 COPY THIS SUMMARY TO YOUR THESIS:")
print(f"   'Classical ML achieved {best_f1:.1%} F1 (best: {best_model}),")
print(f    "establishing baseline for advanced DL approaches'")


SyntaxError: invalid syntax (1823245217.py, line 209)